# RES338 Colab GPU worker

Run this notebook in an already provisioned session with the official `colab exec` CLI. The CLI setup and staging tools pin the source, code, runtime, and private manifests. This notebook exposes only the GPU probe, synthetic qualification, and supervised job launcher; it does not mount Drive or display research rows.

For a supervised operation, pass `RES338_JOB_KIND` or `RES340_JOB_KIND` through `colab exec --env` after staging and setup. RES340 uses a separate campaign folder, the verified 16-cache RES338 archive (`restore-lomo-caches`), a current host Colab balance/rate reading, and an owner execution authorization file. Its fit job advances one frozen fold/configuration/seed at a time with `prepare-lomo` and `fit-lomo-one`. The local controller must persist and verify job evidence in owner-only Drive.

In [ ]:
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/res338")
REPO = ROOT / "repo"
PYTHON = ROOT / "venv" / "bin" / "python"
assert REPO.is_dir() and PYTHON.is_file(), "Run the pinned Colab setup first."
probe = subprocess.run(
    [
        str(PYTHON),
        str(REPO / "tools/res338_colab_gpu_probe.py"),
        "--output",
        str(ROOT / "runtime/notebook-gpu-probe.json"),
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(probe.stdout.strip())

In [ ]:
qualification = subprocess.run(
    [
        str(PYTHON),
        str(REPO / "tools/res338_cuda_qualification.py"),
        "--output",
        str(ROOT / "runtime/notebook-cuda-qualification.json"),
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(qualification.stdout.strip())

In [ ]:
import os
import sys

job_kind = os.environ.get("RES340_JOB_KIND") or os.environ.get("RES338_JOB_KIND")
if job_kind:
    sys.path.insert(0, str(REPO / "tools"))
    from res338_colab_worker import execute

    os.environ["RES338_OPERATION"] = "start"
    print(json.dumps(execute(), sort_keys=True))
else:
    print(
        "Synthetic qualification complete. Set a job kind through colab exec "
        "to start a staged supervised operation."
    )